# 09 — Colour arm at WEAK strength, 12 seeds

The dose-response partner to `color_strong`. `configs/run/color_weak.yaml` is identical
to `color_strong.yaml` in every field except `augmentation.strength`, so the pair isolates
augmentation strength and nothing else.

Purpose: `color_strong` gives a closure fraction of κ < 0 on the hue factors (capacity does
not recover them) and `control_strong` gives κ ≈ 0.74–0.90 (capacity does). A weak-strength
colour arm places a third point between them, turning a two-arm contrast into a
dose-response.

Section 7 packs seeds across both T4s with the subset held in RAM; completed seeds are
skipped, so a timed-out session resumes at the next unfinished one.

**Setup:** Accelerator `GPU T4 x2`, Internet **On**.


## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + build the image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache

## 5. Restore checkpoints from a previous session (resume)
`Add Input -> a prior version's output` (or an uploaded encoder dataset), then run
this. It finds every `color_weak_seed*.pt` under `/kaggle/input` and restores it to
`results/encoders/<run_id>/` (`backbone*.pt` -> `backbone.pt`, `last_ckpt*.pt` ->
`last_ckpt.pt`). On a fresh first run there is nothing to restore. Only scale_recipe
files are touched.

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"color_weak_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)     # first match per (run_id, kind)

if not found:
    print("nothing to restore -- fresh start")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
    print(f"{rid:34s} {tgt:14s} <- {src}")

## 6. Integrity check — keep good checkpoints, purge corrupt ones
Each restored `.pt` is opened as a zip. A valid `backbone.pt` means the seed is
done and training skips it. A corrupt `backbone.pt` is deleted with its
`last_ckpt.pt` so the seed retrains; a valid `last_ckpt.pt` with no backbone lets
training resume mid-run.

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

for d in sorted(ENC.glob("color_weak_seed*")):
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        print(f"{d.name:34s} backbone OK -> skip"); continue
    if bstat != "missing":
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        print(f"{d.name:34s} backbone {bstat} -> purged, will retrain"); continue
    print(f"{d.name:34s} "
          + ("last_ckpt OK -> resume" if _state(ck) is None else "fresh start"))

## 7. Train seeds 0–11 (resumable)

In [ ]:
# Parallel launch (levers 2 + 3 + 4).
#
#   per-gpu 2 x 2 T4s = 4 concurrent runs. Kaggle gives 4 CPU cores, and with
#   data.in_memory the augmentation runs inside each child (run.num_workers=0),
#   so cores, not VRAM, are what bind: 4 concurrent is the sweet spot.
#   RAM cost: subset 150_000 x 64 x 64 x 3 B = 1.84 GB per child, ~7.4 GB for four.
#   The recipe is untouched -- each child is a normal train_simclr run.
#
# Already-finished seeds are skipped, so this cell is safe to re-run after a
# session timeout. Per-seed logs land in results/encoders/_launch_logs/.
import torch
N_GPU = torch.cuda.device_count()
PER_GPU = 2 if N_GPU >= 2 else 3
print(f"{N_GPU} GPU(s) | per-gpu={PER_GPU} | {max(1, N_GPU) * PER_GPU} concurrent")

!cd /kaggle/working/probe-capacity-invariance && python -m src.encoders.launch_parallel \
    --config configs/run/color_weak.yaml \
    --seeds 0 1 2 3 4 5 6 7 8 9 10 11 \
    --per-gpu {PER_GPU} \
    --set data.in_memory=true run.num_workers=0


## 8. Probe the finished cell

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition color --strength weak \
    --encoders results/encoders/color_weak_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --subsample 2000 --probe-steps 1000 \
    --device cuda --num-workers 2 --resume --out-root results/probes_pinned


## 9. Encoder-quality gate — pass rate out of 12 (prereg §5 / D015)

The §5 gate is **three-valued** since A8 §b. `PASS` needs linear-rung shape recoverability
>= 0.90. A failure is only `FAIL-DESTROYED` — the reading that licenses an exclusion — when the
trained-minus-floor anchor deficit is negative at **every** rung and does **not** close with probe
capacity. Otherwise it is `FAIL-INDETERMINATE`: a linear-readout collapse, which is precisely the
ambiguous evidence the capacity ladder exists to disambiguate, and does not license exclusion.

This arm is **EXPLORATORY** (A7 §c). It enters no Holm family and no flip count, and its result is
reported either way — including if it passes, which would mean A5 and A7 excluded their arms for
the wrong reason and would itself require an honest amendment.

In [ ]:
import json
import numpy as np
from pathlib import Path

SHAPE_ABS_MIN = 0.90   # D015, fixed 2026-07-04, blind to every targeted factor
d = Path("results/probes_pinned/color_weak")
z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
names = [f["name"] for f in m["factors"]]
si = names.index("shape")

trained = z["trained"][:, si, :]          # [seeds, rungs]
floor   = z["random"][:, si, :].mean(0)   # [rungs]
lin = trained[:, 0]
n_pass = int((lin >= SHAPE_ABS_MIN).sum())
deficit = trained.mean(0) - floor

print(f"gate pass rate: {n_pass}/{len(lin)}  (threshold {SHAPE_ABS_MIN})")
print(f"linear-rung shape recoverability per seed: "
      f"{np.array2string(np.sort(lin), precision=3)}")
for r, rung in enumerate(m["rungs"]):
    print(f"  {rung:10s} trained {trained[:, r].mean():.3f}  floor {floor[r]:.3f}  "
          f"deficit {deficit[r]:+.3f}")

if n_pass >= 9:
    verdict = "PASS"
elif (deficit < 0).all() and deficit[-1] <= deficit[0]:
    verdict = "FAIL-DESTROYED"
else:
    verdict = "FAIL-INDETERMINATE"
print(f"\nA8 (b) CLASSIFICATION: {verdict}")
print({
 "PASS": "the colour-shortcut account survives: a photometric base recipe rescues the "
         "scale arm, so A5/A7 excluded their arms for the wrong reason. File an honest "
         "amendment saying so. EXPLORATORY - it does not re-enter the confirmatory grid, "
         "which A7 (d) froze at three cells.",
 "FAIL-DESTROYED": "shape information is destroyed, not made nonlinearly inaccessible, even "
         "with photometric augmentation. The construct-mismatch account stands and A7's "
         "exclusion keeps its gate licence.",
 "FAIL-INDETERMINATE": "a linear-readout collapse that closes with capacity. This does NOT "
         "license exclusion on gate grounds and does NOT settle colour-shortcut vs "
         "construct-mismatch: report both accounts as competing, per A7 (c).",
}[verdict])

In [ ]:
# --- persist for the next session --------------------------------------------
# /kaggle/working is the notebook's output. Click "Save Version" when this
# finishes, then Add Input -> this output on the next run to resume.
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")